In [1]:
from dotenv import load_dotenv
import os
load_dotenv()
print(os.getenv("SPARK_LOCAL_IP"))

127.0.0.1


In [2]:
from pyspark.sql import SparkSession

In [3]:
# Create Spark session
# Hadoop AWS connector allows Spark to communicate with Amazon S3
spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("Day-2")
    
    # Use AWS profile credentials from ~/.aws/credentials
    # No access key or secret key stored in code
    .config(
        "spark.hadoop.fs.s3a.aws.credentials.provider",
        "com.amazonaws.auth.profile.ProfileCredentialsProvider"
    )
    
    # S3 connector packages
    .config(
        "spark.jars.packages",
        "org.apache.hadoop:hadoop-aws:3.4.2,"
        "com.amazonaws:aws-java-sdk-bundle:1.12.780"
    )
    
    .getOrCreate()
)


:: loading settings :: url = jar:file:/opt/homebrew/Cellar/apache-spark/4.1.1/libexec/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /Users/rahulsinghrana/.ivy2.5.2/cache
The jars for the packages stored in: /Users/rahulsinghrana/.ivy2.5.2/jars
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-92c5d6e8-ee62-4fac-a89a-313649bb817d;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.4.2 in central
	found software.amazon.awssdk#bundle;2.29.52 in central
	found software.amazon.s3.analyticsaccelerator#analyticsaccelerator-s3;1.2.1 in central
	found org.wildfly.openssl#wildfly-openssl;2.1.4.Final in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.780 in central
:: resolution report :: resolve 136ms :: artifacts dl 5ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.780 from central in [default]
	org.ap

**Task 1**

Read products.csv from your S3 bucket as a DataFrame.

Convert it to an RDD and print the first 3 rows.

What type is each row?

In [4]:
products=spark.read.csv('s3a://pyspark-30-days-rahul-2026/data/products.csv',header=True,inferSchema=True)
products_rdd=products.rdd
for product in products_rdd.take(3):
    print(product)
print('______________________________________________________________________________')
#each product is a Row object,
#  which is similar to a named tuple. 
# we can access the fields of the Row object using dot notation or by index.
#  For example, to access the product name and price, you can do the following:
for product in products_rdd.take(3):
    print(f"Product Name: {product.product_name}, Unit Price: {product.unit_price}")

26/07/29 13:12:26 WARN CredentialProviderListFactory: Credentials option fs.s3a.aws.credentials.provider contains AWS v1 SDK entry com.amazonaws.auth.profile.ProfileCredentialsProvider; mapping to software.amazon.awssdk.auth.credentials.ProfileCredentialsProvider
SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.


Row(product_id='P001', product_name='Laptop Pro 15', category='Electronics', sub_category='Computers', unit_price=1299.99, cost_price=850.0, supplier='TechSupply Co', stock_quantity=150)
Row(product_id='P002', product_name='Wireless Mouse', category='Electronics', sub_category='Accessories', unit_price=29.99, cost_price=12.0, supplier='TechSupply Co', stock_quantity=500)
Row(product_id='P003', product_name='Office Chair Deluxe', category='Furniture', sub_category='Seating', unit_price=349.99, cost_price=180.0, supplier='FurnishPro', stock_quantity=80)
______________________________________________________________________________


Product Name: Laptop Pro 15, Unit Price: 1299.99
Product Name: Wireless Mouse, Unit Price: 29.99
Product Name: Office Chair Deluxe, Unit Price: 349.99


**Task 2**

Create an RDD from this list of tuples: 

[("C001", "James", "Enterprise"), ("C002", "Maria", "SMB"), ("C003", "Robert", "Enterprise")].

Convert it to a DataFrame using toDF() with columns customer_id, first_name, segment. Show the result.

In [5]:
sc=spark.sparkContext
lst=[("C001", "James", "Enterprise"), ("C002", "Maria", "SMB"), ("C003", "Robert", "Enterprise")]
cust_rdd=sc.parallelize(lst)
schema=['customer_id','first_name','segment']
df=cust_rdd.toDF(schema=schema)
df.show()
df.printSchema()

+-----------+----------+----------+
|customer_id|first_name|   segment|
+-----------+----------+----------+
|       C001|     James|Enterprise|
|       C002|     Maria|       SMB|
|       C003|    Robert|Enterprise|
+-----------+----------+----------+

root
 |-- customer_id: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- segment: string (nullable = true)



**Task 3**

Take the same RDD from Task 2 and convert it to a DataFrame using createDataFrame() with an explicit StructType schema. Print the schema using printSchema().

 What is the difference compared to Task 2?

In [6]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

# Define schema explicitly
schema = StructType([
    StructField("customer_id", StringType(), True),
    StructField("first_name", StringType(), True),
    StructField("segment",StringType(),True)
])

# Create DataFrame with explicit schema
cust_df = spark.createDataFrame(cust_rdd, schema)

cust_df.show()
cust_df.printSchema()

+-----------+----------+----------+
|customer_id|first_name|   segment|
+-----------+----------+----------+
|       C001|     James|Enterprise|
|       C002|     Maria|       SMB|
|       C003|    Robert|Enterprise|
+-----------+----------+----------+

root
 |-- customer_id: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- segment: string (nullable = true)

